In [96]:
# Імпортуеємо бібліотеку для імпорту даних
from pycaret.datasets import get_data
import pandas as pd
import numpy as np
# Імпортуємо датасет відповідно до заданого варіанту
df = get_data('automobile')
# Виводимо його форму, сам датасет та наявні поля
df.head(5)
df.columns

,symboling,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,...,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
0,3,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495
1,3,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500
2,1,NaN,alfa-romero,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500
3,2,164.0,audi,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.4,10.0,102,5500,24,30,13950
4,2,164.0,audi,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.4,8.0,115,5500,18,22,17450


Index(['symboling', 'normalized-losses', 'make', 'fuel-type', 'aspiration',
       'num-of-doors', 'body-style', 'drive-wheels', 'engine-location',
       'wheel-base', 'length', 'width', 'height', 'curb-weight', 'engine-type',
       'num-of-cylinders', 'engine-size', 'fuel-system', 'bore', 'stroke',
       'compression-ratio', 'horsepower', 'peak-rpm', 'city-mpg',
       'highway-mpg', 'price'],
      dtype='object')

In [97]:
# Перетворюємо певні категоріальні поля, які мають бути числовими, у числові
df['horsepower'] = pd.to_numeric(df['horsepower'], errors="coerce")
df['bore'] = pd.to_numeric(df['bore'], errors="coerce")
df['stroke'] = pd.to_numeric(df['stroke'], errors="coerce")
df['peak-rpm'] = pd.to_numeric(df['peak-rpm'], errors="coerce")

In [98]:
from pycaret.clustering import *
# Ініціюємо модель з попередньою нормалізацією даних
cl = setup(df, normalize = True, session_id = 42)
cl

,Description,Value
0,Session id,42
1,Original data shape,"(202, 26)"
2,Transformed data shape,"(202, 71)"
3,Ordinal features,4
4,Numeric features,16
5,Categorical features,10
6,Rows with missing values,20.8%
7,Preprocess,True
8,Imputation type,simple
9,Numeric imputation,mean


In [99]:
# Візьмемо метод к найближчих сусідів та кількість кластерів - 2
kmeans = create_model('kmeans', num_clusters = 2)


,Silhouette,Calinski-Harabasz,Davies-Bouldin,Homogeneity,Rand Index,Completeness
0,0.1629,27.6112,2.4893,0,0,0


In [100]:
# Результат кластерізації
kmean_results = assign_model(kmeans)
kmean_results

,symboling,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,...,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price,Cluster
0,3,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.599998,...,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,13495,Cluster 0
1,3,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.599998,...,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,16500,Cluster 0
2,1,NaN,alfa-romero,gas,std,two,hatchback,rwd,front,94.500000,...,mpfi,2.68,3.47,9.0,154.0,5000.0,19,26,16500,Cluster 0
3,2,164.0,audi,gas,std,four,sedan,fwd,front,99.800003,...,mpfi,3.19,3.40,10.0,102.0,5500.0,24,30,13950,Cluster 1
4,2,164.0,audi,gas,std,four,sedan,4wd,front,99.400002,...,mpfi,3.19,3.40,8.0,115.0,5500.0,18,22,17450,Cluster 0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
197,-1,95.0,volvo,gas,turbo,four,sedan,rwd,front,109.099998,...,mpfi,3.78,3.15,8.7,160.0,5300.0,19,25,19045,Cluster 0
198,-1,95.0,volvo,gas,std,four,sedan,rwd,front,109.099998,...,mpfi,3.58,2.87,8.8,134.0,5500.0,18,23,21485,Cluster 0
199,-1,95.0,volvo,diesel,turbo,four,sedan,rwd,front,109.099998,...,idi,3.01,3.40,23.0,106.0,4800.0,26,27,22470,Cluster 0
200,-1,95.0,volvo,gas,turbo,four,sedan,rwd,front,109.099998,...,mpfi,3.78,3.15,9.5,114.0,5400.0,19,25,22625,Cluster 0


In [101]:
# Графік кластерів
plot_model(kmeans)

In [102]:
# Графік розподілу кількості записів у кожному кластері
plot_model(kmeans, plot = 'distribution')

In [109]:
# Нижче для всього датасету та для кожного кластеру виводимо середні значення для всіх числових даних 
cluster_1 = kmean_results[kmean_results["Cluster"] == f"Cluster {0}"]
cluster_2 = kmean_results[kmean_results["Cluster"] == f"Cluster {1}"]
for feature in df.select_dtypes(include=np.number).columns.tolist():
    print(f"  Mean {feature} for dataset: {df[feature].mean()}")
    print(f"  Mean {feature} for cluster 0: {cluster_1[feature].mean()}")
    print(f"  Mean {feature} for cluster 1: {cluster_2[feature].mean()}\n")

  Mean symboling for dataset: 0.8316831683168316
  Mean symboling for cluster 0: 0.4714285714285714
  Mean symboling for cluster 1: 1.0227272727272727

  Mean normalized-losses for dataset: 121.83636363636364
  Mean normalized-losses for cluster 0: 136.21739196777344
  Mean normalized-losses for cluster 1: 116.27731323242188

  Mean wheel-base for dataset: 98.84801980198021
  Mean wheel-base for cluster 0: 104.10001373291016
  Mean wheel-base for cluster 1: 96.06288146972656

  Mean length for dataset: 174.27326732673265
  Mean length for cluster 0: 185.85569763183594
  Mean length for cluster 1: 168.13107299804688

  Mean width for dataset: 65.9039603960396
  Mean width for cluster 0: 67.91427612304688
  Mean width for cluster 1: 64.83787536621094

  Mean height for dataset: 53.77524752475248
  Mean height for cluster 0: 54.68143081665039
  Mean height for cluster 1: 53.294700622558594

  Mean curb-weight for dataset: 2558.1732673267325
  Mean curb-weight for cluster 0: 3116.657142857

In [103]:

# Виводимо графік розподілу за кількістю дверей у авто
plot_model(kmeans, plot = 'distribution', feature="num-of-doors")

In [104]:

# Графік розподілу за системою пального
plot_model(kmeans, plot = 'distribution', feature="fuel-system")

In [105]:

# Графік розподілу за типом двигуна
plot_model(kmeans, plot = 'distribution', feature="engine-type")

In [106]:

# Графік розподілу за кількістю циліндрів двигуна
plot_model(kmeans, plot = 'distribution', feature="num-of-cylinders")

In [110]:
# Графік розподілу за типом кузова
plot_model(kmeans, plot = 'distribution', feature="body-style")

In [111]:
# Графік розподілу за типом ведучих коліс
plot_model(kmeans, plot = 'distribution', feature="drive-wheels")

In [112]:
# Зберігаємо модель
save_model(kmeans, model_name='kmeans_model')

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=FastMemory(location=C:\Users\karda\AppData\Local\Temp\joblib),
          steps=[('numerical_imputer',
                  TransformerWrapper(include=['symboling', 'normalized-losses',
                                              'wheel-base', 'length', 'width',
                                              'height', 'curb-weight',
                                              'engine-size', 'bore', 'stroke',
                                              'compression-ratio', 'horsepower',
                                              'peak-rpm', 'city-mpg',
                                              'highway-mpg', 'price'],
                                     transformer=SimpleImput...
                  TransformerWrapper(include=['make', 'body-style',
                                              'drive-wheels', 'engine-type',
                                              'num-of-cylinders',
                                              'fuel-system'],
             

Опираючись на усі дані,  що були виведені вище, можемо зазначити, що дана кластерізація вийшла досить збалансованою. Враховуючи те, що даний датасет складався тільки з 213 записів ми отримали приблизне співвідношення 2:1 у кількості екземплярів кожного кластеру.Слід зазначити, що для більшої кількості кластерів, через розмір нашої вибірки, кластери стають досить малими, і через це втрачається практична цінність результатів, тому кількість кластерів - 2.


Якщо подивитися на такі параметри як розмір авто (length, width, height), масу, об'єм двигуна, ціну та інші параметри, можемо з упевненіст сказати, що в нульовому кластері ми отримали більш великі, потужні та дорожчі автомобілі ніж у першому. Це відображається не тільки в числових даних, а й в категоріальних, адже, як можна побачити на графіках, нульовий кластер зазвичай має більшу кількість ціліндрів двигуна, зазвичай має задній привід, має тільки новітні системи подачі пального та інше.
